# Florence-2 OCR fine-tune

Loads the private Hugging Face dataset `tejaskale/florence-handwriting-ocr`. Set `HF_TOKEN` in Colab secrets.

In [ ]:
!pip install -q transformers timm einops accelerate datasets pillow huggingface_hub

In [ ]:
import random, torch
from datasets import load_dataset; from google.colab import userdata
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoProcessor, get_scheduler

In Colab, add a secret named `HF_TOKEN` with read access to the private dataset.

In [ ]:
model_id = "microsoft/Florence-2-base-ft"
revision = "refs/pr/6"
dataset_id = "tejaskale/florence-handwriting-ocr"
out_dir = Path("/content/florence-ocr-ft")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN in Colab secrets and allow notebook access"
items = list(load_dataset(dataset_id, token=hf_token)["train"])
split = max(1, int(len(items) * 0.8))
train_items, val_items = items[:split], items[split:]
len(items)

In [ ]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, revision=revision)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, revision=revision).to(device)
torch.cuda.empty_cache()

In [ ]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, revision=revision)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, revision=revision).to(device)
torch.cuda.empty_cache()

In [ ]:
for p in model.vision_tower.parameters():
    p.requires_grad = False

In [ ]:
task = "<OCR>"

In [ ]:
def collate(batch):
    images = [x["image"].convert("RGB") for x in batch]
    x = processor(text=[task] * len(batch), images=images, return_tensors="pt", padding=True).to(device)
    y = processor.tokenizer([x["text"] for x in batch], return_tensors="pt", padding=True).input_ids.to(device)
    y[y == processor.tokenizer.pad_token_id] = -100
    x["labels"] = y; return x

In [ ]:
train_loader = DataLoader(train_items, batch_size=1, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_items, batch_size=1, collate_fn=collate)
len(train_loader), len(val_loader)

In [ ]:
epochs, lr = 3, 1e-6
optim = torch.optim.AdamW(model.parameters(), lr=lr)
steps = epochs * len(train_loader)
sched = get_scheduler("linear", optim, 0, steps)
model.train()

In [ ]:
for _ in range(epochs):
    for batch in train_loader:
        loss = model(**batch).loss
        loss.backward()
        optim.step(); sched.step(); optim.zero_grad()
last_loss = loss.detach().float().cpu()

In [ ]:
model.eval()
losses = []
with torch.no_grad():
    for batch in val_loader:
        losses.append(model(**batch).loss.detach().float().cpu())
sum(losses) / len(losses) if losses else last_loss

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(out_dir)
processor.save_pretrained(out_dir)
out_dir

Run OCR on a random training image as a smoke test.

In [ ]:
ft_processor = AutoProcessor.from_pretrained(out_dir, trust_remote_code=True)
ft_model = AutoModelForCausalLM.from_pretrained(out_dir, trust_remote_code=True).to(device)
ft_model.eval()

In [ ]:
sample = random.choice(train_items)
image = sample["image"].convert("RGB")
image

In [ ]:
inputs = ft_processor(text=task, images=image, return_tensors="pt").to(device)
ids = ft_model.generate(**inputs, max_new_tokens=128)
predicted = ft_processor.batch_decode(ids, skip_special_tokens=True)[0]
{"expected": sample["text"], "predicted": predicted}